# ADD 2026 Track 2 - Qwen2.5-Omni Baseline Dev Evaluation

This notebook downloads the baseline code archive, released dataset, and official checkpoint from Google Drive, runs a smoke test, performs restart-safe sharded inference on all 2,000 Dev samples, and evaluates Official Dev.

Start with `MAX_BATCH_SIZE=1`; increase only after the smoke test succeeds. This notebook uses the Python 3.11 environment already configured in Colab and installs the pinned project requirements.

All persistent outputs are written to Google Drive. The downloaded dataset must not be redistributed.

## 1. Configuration
Edit only this cell when paths or inference settings change.

In [ ]:
from pathlib import Path, PurePosixPath
import os, sys, json, time, shutil, zipfile, re

PROJECT_DRIVE_FILE_ID = "1WhUdcUB6dDW6fKtcMqDaF-GNzGwi1abJ"
DATASET_DRIVE_FILE_ID = "1QQIEf5lE5I5dIpKrnxGW8noj71zxZmd8"
CHECKPOINT_DRIVE_FILE_ID = "1_Tr_Bpm5mmTyIQi2Px7ATyXDPENqXIxp"

CONTENT_ROOT = Path("/content")
PROJECT_DIR = CONTENT_ROOT / "track2_baseline"
WORK_DIR = PROJECT_DIR / ".colab_work"
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/ADD2026/results/qwen25_baseline_dev")

PROJECT_DOWNLOAD = CONTENT_ROOT / "track2_baseline.zip"
DATASET_DOWNLOAD = CONTENT_ROOT / "track2_training_datasets.zip"
CHECKPOINT_DOWNLOAD = CONTENT_ROOT / "track2_baseline_weights.zip"
SHARD_SIZE = 500
SMOKE_SAMPLES = 8
MAX_BATCH_SIZE = 1
MAX_NEW_TOKENS = 2048

print({
    "project_file_id": PROJECT_DRIVE_FILE_ID,
    "dataset_file_id": DATASET_DRIVE_FILE_ID,
    "checkpoint_file_id": CHECKPOINT_DRIVE_FILE_ID,
    "results_dir": str(DRIVE_RESULTS_DIR),
    "shard_size": SHARD_SIZE,
    "max_batch_size": MAX_BATCH_SIZE,
})

## 2. Mount Drive
Google Drive is used only for persistent evaluation outputs.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Check the existing Colab environment
This cell verifies the Python 3.11 and gdown commands required to download the project.

In [ ]:
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Expected Python 3.11, found {sys.version.split()[0]}")
PYTHON_BIN = shutil.which("python")
GDOWN_BIN = shutil.which("gdown")
if not PYTHON_BIN or not GDOWN_BIN:
    raise RuntimeError({"python": PYTHON_BIN, "gdown": GDOWN_BIN})
print({"python": PYTHON_BIN, "gdown": GDOWN_BIN})
!python --version

## 4. Download the code and dataset archives
The Drive file `1WhU...` is a small `track2_baseline.zip` containing the baseline code. The released `train/`, `dev/`, and `labels/` data are downloaded separately from `1QQI...`. The dataset archive is extracted to Colab's local SSD; only results are written back to your Drive.

In [ ]:
project_url = f"https://drive.google.com/file/d/{PROJECT_DRIVE_FILE_ID}/view"
if not PROJECT_DOWNLOAD.exists():
    !gdown --fuzzy {project_url} -O {PROJECT_DOWNLOAD}
else:
    print("Reusing downloaded project archive:", PROJECT_DOWNLOAD)

if not PROJECT_DOWNLOAD.is_file() or PROJECT_DOWNLOAD.stat().st_size == 0:
    raise RuntimeError("Project archive download failed or is empty.")

if not (PROJECT_DIR / "prepare_data.py").is_file():
    if not zipfile.is_zipfile(PROJECT_DOWNLOAD):
        raise RuntimeError("The downloaded project file is not a valid ZIP archive.")
    with zipfile.ZipFile(PROJECT_DOWNLOAD) as zf:
        zf.extractall(CONTENT_ROOT)
if not (PROJECT_DIR / "prepare_data.py").is_file():
    raise RuntimeError(f"prepare_data.py not found under {PROJECT_DIR}")
print("Project root:", PROJECT_DIR)
print("Project archive size (MiB):", round(PROJECT_DOWNLOAD.stat().st_size / 2**20, 3))

def progress(done, total, start, label):
    elapsed = max(time.time() - start, 1e-6)
    eta = elapsed * (total - done) / done if done else 0
    print(f"[{label}] {done}/{total} ({100 * done / total:.1f}%) elapsed={elapsed:.1f}s ETA={eta:.1f}s", flush=True)

dataset_url = f"https://drive.google.com/file/d/{DATASET_DRIVE_FILE_ID}/view"
if not DATASET_DOWNLOAD.exists():
    !gdown --fuzzy {dataset_url} -O {DATASET_DOWNLOAD}
else:
    print("Reusing downloaded dataset archive:", DATASET_DOWNLOAD)
if not DATASET_DOWNLOAD.is_file() or DATASET_DOWNLOAD.stat().st_size == 0:
    raise RuntimeError("Dataset archive download failed or is empty.")

required_parts = ("train", "dev", "labels")
existing_data = PROJECT_DIR / "data"
existing_data.mkdir(parents=True, exist_ok=True)
for part in required_parts:
    target = existing_data / part
    if target.is_symlink():
        target.unlink()

data_ready = (
    len(list((existing_data / "train").glob("*.wav"))) == 8600
    and len(list((existing_data / "dev").glob("*.wav"))) == 2000
    and (existing_data / "labels/train_labels.jsonl").is_file()
    and (existing_data / "labels/dev_labels.jsonl").is_file()
)
if not data_ready:
    if not zipfile.is_zipfile(DATASET_DOWNLOAD):
        raise RuntimeError("The downloaded dataset is not a valid ZIP archive. Check Drive access/quota.")
    with zipfile.ZipFile(DATASET_DOWNLOAD) as zf:
        selected = []
        for member in zf.infolist():
            parts = PurePosixPath(member.filename).parts
            if "track2_training_datasets" not in parts:
                continue
            offset = parts.index("track2_training_datasets") + 1
            relative_parts = parts[offset:]
            if relative_parts and relative_parts[0] in required_parts and ".." not in relative_parts:
                selected.append((member, relative_parts))
        if not selected:
            raise RuntimeError("No track2_training_datasets/train, dev, or labels entries found in dataset ZIP.")
        start = time.time()
        for index, (member, relative_parts) in enumerate(selected, 1):
            destination = existing_data.joinpath(*relative_parts)
            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as source, destination.open("wb") as output:
                    shutil.copyfileobj(source, output)
            if index % 500 == 0 or index == len(selected):
                progress(index, len(selected), start, "extract-data")
print("Dataset root:", existing_data)
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 5. Validate the released dataset before installing the model stack

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8-sig").splitlines() if line.strip()]

expected = {"train": 8600, "dev": 2000}
for split, count in expected.items():
    wavs = sorted((PROJECT_DIR / "data" / split).glob("*.wav"))
    labels = read_jsonl(PROJECT_DIR / "data" / "labels" / f"{split}_labels.jsonl")
    wav_ids = {p.stem for p in wavs}
    label_ids = {r["file_id"] for r in labels}
    assert len(wavs) == count, (split, len(wavs), count)
    assert len(labels) == count, (split, len(labels), count)
    assert wav_ids == label_ids, f"{split}: WAV/label ID mismatch"
    counts = {label: sum(r["label"] == label for r in labels) for label in ("real", "fake")}
    print(split, {"wavs": len(wavs), "labels": len(labels), **counts})

## 6. Install and validate the model environment
Install the project's pinned Python requirements into the existing Colab Python 3.11 environment, then verify ms-swift before checkpoint download and inference.

In [ ]:
%cd {PROJECT_DIR}
!python -m pip install -q -r requirements.txt
SWIFT_BIN = shutil.which("swift")
if not SWIFT_BIN:
    raise RuntimeError("ms-swift installation completed but the swift CLI was not found.")
os.environ["PYTHONUNBUFFERED"] = "1"
!python -c "import torch, transformers, swift, bert_score; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())"

## 7. Download, verify, and extract the official baseline checkpoint

In [ ]:
checkpoint_url = f"https://drive.google.com/file/d/{CHECKPOINT_DRIVE_FILE_ID}/view"
if not CHECKPOINT_DOWNLOAD.exists():
    !gdown --fuzzy {checkpoint_url} -O {CHECKPOINT_DOWNLOAD}
else:
    print("Reusing downloaded checkpoint archive:", CHECKPOINT_DOWNLOAD)

if not zipfile.is_zipfile(CHECKPOINT_DOWNLOAD):
    raise RuntimeError("Checkpoint download is not a valid ZIP archive. Check Drive access/quota.")

CHECKPOINT_EXTRACT_ROOT = PROJECT_DIR
expected_checkpoint = PROJECT_DIR / "track2_baseline_weights"
if not expected_checkpoint.exists():
    with zipfile.ZipFile(CHECKPOINT_DOWNLOAD) as zf:
        zf.extractall(CHECKPOINT_EXTRACT_ROOT)

config_candidates = sorted(expected_checkpoint.rglob("config.json"))
weight_roots = [p.parent for p in config_candidates if list(p.parent.glob("*.safetensors")) or list(p.parent.glob("*.bin"))]
if len(weight_roots) != 1:
    raise RuntimeError(f"Expected one checkpoint directory, found: {weight_roots}")
CHECKPOINT_DIR = weight_roots[0]
print("Checkpoint directory:", CHECKPOINT_DIR)

## 8. Build ms-swift manifests and run a smoke test
The smoke test uses eight Dev files. Do not start the full run unless it produces eight valid XML responses without OOM.

In [ ]:
%cd {PROJECT_DIR}
!{PYTHON_BIN} prepare_data.py
swift_dev = read_jsonl(PROJECT_DIR / "data/swift/dev.jsonl")
assert len(swift_dev) == 2000
print("Prepared Dev manifest:", len(swift_dev))

In [ ]:
SMOKE_AUDIO_DIR = WORK_DIR / "smoke_wavs"
SMOKE_OUTPUT_DIR = WORK_DIR / "smoke_output"
SMOKE_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
for wav in sorted((PROJECT_DIR / "data/dev").glob("*.wav"))[:SMOKE_SAMPLES]:
    link = SMOKE_AUDIO_DIR / wav.name
    if not link.exists():
        link.symlink_to(wav)

raw_smoke = SMOKE_OUTPUT_DIR / "raw_predictions.jsonl"
if raw_smoke.exists() and sum(1 for line in raw_smoke.open() if line.strip()) == SMOKE_SAMPLES:
    print("Smoke test already complete; skipping inference.")
else:
    if SMOKE_OUTPUT_DIR.exists():
        archived = SMOKE_OUTPUT_DIR.with_name(f"smoke_output.partial_{int(time.time())}")
        SMOKE_OUTPUT_DIR.rename(archived)
        print("Archived incomplete smoke output to", archived)
    os.environ["CHECKPOINT_DIR"] = str(CHECKPOINT_DIR)
    os.environ["SMOKE_AUDIO_DIR"] = str(SMOKE_AUDIO_DIR)
    os.environ["SMOKE_OUTPUT_DIR"] = str(SMOKE_OUTPUT_DIR)
    os.environ["MAX_BATCH_SIZE"] = str(MAX_BATCH_SIZE)
    os.environ["MAX_NEW_TOKENS"] = str(MAX_NEW_TOKENS)
    !bash infer.sh "$CHECKPOINT_DIR" "$SMOKE_AUDIO_DIR" "$SMOKE_OUTPUT_DIR"

smoke_rows = read_jsonl(raw_smoke)
assert len(smoke_rows) == SMOKE_SAMPLES
valid_xml = sum(
    len(re.findall(r"<Decision>\s*(?:real|fake)\s*</Decision>", row.get("response", ""), re.I)) == 1
    and len(re.findall(r"<Evidence>.*?</Evidence>", row.get("response", ""), re.I | re.S)) == 1
    for row in smoke_rows
)
print({"smoke_rows": len(smoke_rows), "valid_xml": valid_xml})
if valid_xml != SMOKE_SAMPLES:
    raise RuntimeError("Smoke test completed but not every response contains exactly one Decision and Evidence field.")

In [ ]:
official_by_id = {row["id"]: row for row in swift_dev}
smoke_subset_rows = []
for prediction in smoke_rows:
    smoke_audio = prediction["audios"][0]
    file_id = Path(smoke_audio).stem
    reference = dict(official_by_id[file_id])
    reference["audios"] = [smoke_audio]
    smoke_subset_rows.append(reference)

SMOKE_SUBSET = WORK_DIR / "smoke_subset.jsonl"
SMOKE_ACC = WORK_DIR / "smoke_accuracy.json"
SMOKE_EVAL = WORK_DIR / "smoke_eval.json"
SMOKE_SUBSET.write_text(
    "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in smoke_subset_rows),
    encoding="utf-8",
)
for name, value in {
    "SMOKE_RAW": raw_smoke,
    "SMOKE_SUBSET": SMOKE_SUBSET,
    "SMOKE_ACC": SMOKE_ACC,
    "SMOKE_EVAL": SMOKE_EVAL,
}.items():
    os.environ[name] = str(value)

!{PYTHON_BIN} eval_add2026.py "$SMOKE_RAW" --subset "$SMOKE_SUBSET" --no-bert --out-json "$SMOKE_ACC"
!{PYTHON_BIN} eval_add2026.py "$SMOKE_RAW" --subset "$SMOKE_SUBSET" --bert-batch-size 8 --bert-device cuda:0 --out-json "$SMOKE_EVAL"

smoke_eval = json.loads(SMOKE_EVAL.read_text(encoding="utf-8"))
assert smoke_eval["subset_total"] == SMOKE_SAMPLES
assert smoke_eval["matched_predictions"] == SMOKE_SAMPLES
assert smoke_eval["missing_predictions"] == 0
assert smoke_eval["bertscore_pairs"] == SMOKE_SAMPLES
assert smoke_eval["final_score_100"] is not None
print(json.dumps({
    "samples": smoke_eval["subset_total"],
    "balanced_accuracy": smoke_eval["balanced_accuracy"],
    "joint_score": smoke_eval["joint_score"],
    "final_score_100": smoke_eval["final_score_100"],
}, ensure_ascii=False, indent=2))

## 9. Create restart-safe Dev shards
Four 500-file shards are created with symbolic links. Completed shard outputs in Drive are skipped on rerun.

In [ ]:
SHARDS_ROOT = WORK_DIR / "dev_shards"
SHARDS_ROOT.mkdir(parents=True, exist_ok=True)
dev_wavs = sorted((PROJECT_DIR / "data/dev").glob("*.wav"))
num_shards = (len(dev_wavs) + SHARD_SIZE - 1) // SHARD_SIZE
for shard_index in range(num_shards):
    shard_dir = SHARDS_ROOT / f"shard_{shard_index:03d}"
    shard_dir.mkdir(exist_ok=True)
    members = dev_wavs[shard_index * SHARD_SIZE:(shard_index + 1) * SHARD_SIZE]
    for wav in members:
        link = shard_dir / wav.name
        if not link.exists():
            link.symlink_to(wav)
    assert len(list(shard_dir.glob("*.wav"))) == len(members)
    print(shard_dir.name, len(members))

os.environ.update({
    "ADD_PROJECT_DIR": str(PROJECT_DIR),
    "ADD_CHECKPOINT_DIR": str(CHECKPOINT_DIR),
    "ADD_SHARDS_ROOT": str(SHARDS_ROOT),
    "ADD_RESULTS_DIR": str(DRIVE_RESULTS_DIR),
    "ADD_NUM_SHARDS": str(num_shards),
    "ADD_MAX_BATCH_SIZE": str(MAX_BATCH_SIZE),
    "ADD_MAX_NEW_TOKENS": str(MAX_NEW_TOKENS),
})

## 10. Run full Dev inference
This is the long-running cell. ms-swift displays batch progress; the wrapper additionally reports shard completion, throughput, and updated ETA. Incomplete outputs are preserved with a `.partial_TIMESTAMP` suffix before retrying.

In [ ]:
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MAX_BATCH_SIZE"] = str(MAX_BATCH_SIZE)
os.environ["MAX_NEW_TOKENS"] = str(MAX_NEW_TOKENS)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
%cd {PROJECT_DIR}
overall_start = time.time()

for shard_index in range(num_shards):
    shard = f"shard_{shard_index:03d}"
    audio_dir = SHARDS_ROOT / shard
    output_dir = DRIVE_RESULTS_DIR / shard
    raw_predictions = output_dir / "raw_predictions.jsonl"
    expected = len(list(audio_dir.glob("*.wav")))
    actual = sum(1 for line in raw_predictions.open(encoding="utf-8") if line.strip()) if raw_predictions.is_file() else 0

    if actual == expected:
        print(f"[skip] {shard} already complete ({actual}/{expected})", flush=True)
    else:
        if output_dir.exists():
            archived = output_dir.with_name(f"{output_dir.name}.partial_{int(time.time())}")
            output_dir.rename(archived)
            print(f"[archive] {output_dir} -> {archived}", flush=True)
        os.environ["CURRENT_AUDIO_DIR"] = str(audio_dir)
        os.environ["CURRENT_OUTPUT_DIR"] = str(output_dir)
        shard_start = time.time()
        print(f"[start] {shard} ({expected} samples) at {time.strftime('%Y-%m-%dT%H:%M:%S')}", flush=True)
        !bash infer.sh "$ADD_CHECKPOINT_DIR" "$CURRENT_AUDIO_DIR" "$CURRENT_OUTPUT_DIR"
        elapsed = time.time() - shard_start
        actual = sum(1 for line in raw_predictions.open(encoding="utf-8") if line.strip()) if raw_predictions.is_file() else 0
        if actual != expected:
            raise RuntimeError(f"{shard} produced {actual}/{expected} rows")
        print(f"[done] {shard} elapsed={elapsed:.1f}s rate={actual / elapsed:.4f} samples/s", flush=True)

    completed = shard_index + 1
    elapsed_total = time.time() - overall_start
    remaining = num_shards - completed
    eta = elapsed_total * remaining / completed
    print(f"[progress] shards={completed}/{num_shards} elapsed={elapsed_total:.1f}s ETA={eta:.1f}s", flush=True)

## 11. Merge shards and normalize audio keys
Shard paths are rewritten to canonical `data/dev/<file>.wav` paths so `eval_add2026.py` can match all predictions to the official Dev manifest.

In [ ]:
merged_inputs = []
merged_raw = []
for shard_index in range(num_shards):
    shard_output = DRIVE_RESULTS_DIR / f"shard_{shard_index:03d}"
    inputs = read_jsonl(shard_output / "input.jsonl")
    raw = read_jsonl(shard_output / "raw_predictions.jsonl")
    assert len(inputs) == len(raw)
    for row in inputs:
        fid = row.get("id") or Path(row["audios"][0]).stem
        row["id"] = fid
        row["audios"] = [f"data/dev/{fid}.wav"]
    for row in raw:
        fid = Path(row["audios"][0]).stem
        row["audios"] = [f"data/dev/{fid}.wav"]
    merged_inputs.extend(inputs)
    merged_raw.extend(raw)

def write_jsonl(path, rows):
    path = Path(path)
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")

input_ids = [r["id"] for r in merged_inputs]
raw_ids = [Path(r["audios"][0]).stem for r in merged_raw]
assert len(merged_inputs) == len(merged_raw) == 2000
assert len(set(input_ids)) == len(set(raw_ids)) == 2000
assert set(input_ids) == set(raw_ids)

MERGED_INPUT = DRIVE_RESULTS_DIR / "input.jsonl"
MERGED_RAW = DRIVE_RESULTS_DIR / "raw_predictions.jsonl"
PREDICTION_JSONL = DRIVE_RESULTS_DIR / "prediction.jsonl"
write_jsonl(MERGED_INPUT, merged_inputs)
write_jsonl(MERGED_RAW, merged_raw)
print({"merged_inputs": len(merged_inputs), "merged_predictions": len(merged_raw)})

os.environ["MERGED_INPUT"] = str(MERGED_INPUT)
os.environ["MERGED_RAW"] = str(MERGED_RAW)
os.environ["PREDICTION_JSONL"] = str(PREDICTION_JSONL)
!{PYTHON_BIN} export_submission.py --input "$MERGED_INPUT" --raw "$MERGED_RAW" --output "$PREDICTION_JSONL"

## 12. Evaluate Official Dev
Classification-only metrics run first. BERTScore then runs on GPU after the inference process has released model memory.

In [ ]:
OFFICIAL_SUBSET = PROJECT_DIR / "data/swift/dev.jsonl"
official_rows = read_jsonl(OFFICIAL_SUBSET)
assert len(official_rows) == 2000

OFFICIAL_ACC = DRIVE_RESULTS_DIR / "official_dev_accuracy.json"
OFFICIAL_EVAL = DRIVE_RESULTS_DIR / "official_dev_eval.json"
for name, value in {
    "OFFICIAL_SUBSET": OFFICIAL_SUBSET,
    "OFFICIAL_ACC": OFFICIAL_ACC,
    "OFFICIAL_EVAL": OFFICIAL_EVAL,
}.items():
    os.environ[name] = str(value)

!{PYTHON_BIN} eval_add2026.py "$MERGED_RAW" --subset "$OFFICIAL_SUBSET" --no-bert --out-json "$OFFICIAL_ACC"
!{PYTHON_BIN} eval_add2026.py "$MERGED_RAW" --subset "$OFFICIAL_SUBSET" --bert-device cuda:0 --out-json "$OFFICIAL_EVAL"

## 13. Final validation and experiment summary

In [ ]:
official = json.loads(OFFICIAL_EVAL.read_text(encoding="utf-8"))
responses = [row.get("response", "") for row in merged_raw]
xml_valid = [
    len(re.findall(r"<Decision>\s*(?:real|fake)\s*</Decision>", response, re.I)) == 1
    and len(re.findall(r"<Evidence>.*?</Evidence>", response, re.I | re.S)) == 1
    for response in responses
]
evidence_words = []
for response in responses:
    match = re.search(r"<Evidence>(.*?)</Evidence>", response, re.I | re.S)
    text = re.sub(r"<[^>]+>", " ", match.group(1)) if match else ""
    evidence_words.append(len(re.sub(r"\s+", " ", text).strip().split()))

summary = {
    "checkpoint_drive_file_id": CHECKPOINT_DRIVE_FILE_ID,
    "project_drive_file_id": PROJECT_DRIVE_FILE_ID,
    "dataset_drive_file_id": DATASET_DRIVE_FILE_ID,
    "official_dev": {
        "samples": official["subset_total"],
        "real_recall": official["real"]["recall"],
        "fake_recall": official["fake"]["recall"],
        "balanced_accuracy": official["balanced_accuracy"],
        "overall_accuracy": official["overall_accuracy"],
        "mean_bertscore_f1": (
            official["real"]["mean_bertscore_f1_all"] * official["real"]["total"]
            + official["fake"]["mean_bertscore_f1_all"] * official["fake"]["total"]
        ) / official["subset_total"],
        "real_joint": official["real"]["joint"],
        "fake_joint": official["fake"]["joint"],
        "final_score_100": official["final_score_100"],
    },
    "xml_validity_rate": sum(xml_valid) / len(xml_valid),
    "mean_evidence_words": sum(evidence_words) / len(evidence_words),
    "max_batch_size": MAX_BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
}
SUMMARY_PATH = DRIVE_RESULTS_DIR / "summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Saved all persistent artifacts to:", DRIVE_RESULTS_DIR)